In [4]:
import pandas as pd
import numpy as np
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

df = pd.read_csv('C:\\FUFA Code\\Match-Analysis\\data\\raw\\FWSL25_26_midseason_raw_catapult_data.csv')
# df=pd.read_csv('C:\\Users\\Travail\\Downloads\\Catapult-Export-1769603622121.csv')

In [5]:
df.shape

(4868, 100)

In [6]:
# strip whitespace, replace spaces/slashes with underscores
df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(r'\s+', '_', regex=True)
      .str.replace(r'[\/\(\)\-]', '', regex=True)
)

In [7]:
def clean_text(s):
    if pd.isnull(s):
        return s
    s = s.strip()                        # Remove leading/trailing whitespace
    s = re.sub(r'\s+', ' ', s)          # Replace multiple spaces with single
    s = s.lower()                       # Convert to lowercase (optional)
    # s = s.replace('_', ' ')             # Replace underscores with space
    s = s.title()                       # Convert to Title Case

    return s

# Apply cleaning to each text column
cat_cols = [col for col in df.columns if df[col].dtype == 'object']
for col in cat_cols:
    df[col] = df[col].astype(str).apply(clean_text)

In [8]:
#convert date to datetime format
df['date'] = pd.to_datetime(df['date'], dayfirst=True, errors='coerce')

In [9]:
#select only the 'Game' tag, exclude training etc 
df = df[((df['tags'] == 'Game') | (df['tags'] == 'Game Training'))] #game training is a system glitch but also covers game data 
df = df.drop('tags', axis=1)

In [10]:
df['duration'] = df['duration'] / 60 #convert duration to minutes for easy comparison

In [11]:
df.shape

(3107, 99)

In [12]:
# #select only the first and second halves ; ensure proper naming of splits

# df[(df['split_name'] == '1St.Half') | (df['split_name'] == '2Nd.Half')] = df
# df['split_name'] = df['split_name'].str.replace('1St.Half', '1st Half')
# df['split_name'] = df['split_name'].str.replace('2Nd.Half', '2nd Half')

In [13]:
#select only the first and second halves ; ensure proper naming of splits
df = df[((df['split_name'] == '1St.Half') | (df['split_name'] == '2Nd.Half') | (df['split_name'] == 'All'))]
df['split_name'] = df['split_name'].str.replace('1St.Half', '1st Half')
df['split_name'] = df['split_name'].str.replace('2Nd.Half', '2nd Half')

In [14]:
df.shape

(2720, 99)

In [15]:
# # Select only session_title entries matching the pattern: 'Wmd1-Kampala Queens Fc-Olila HS WFC-Home-League-Win' ( standard naming convention)
# pattern = r'^Wmd\d+-[\w\s\.]+-[\w\s\.]+-[\w\s\.]+-[\w\s\.]+-[\w\s\.]+$'
# df = df[df['session_title'].str.match(pattern, na=False)]

# Select only session_title entries matching the pattern: 'Md1-Kampala Queens Fc-Olila HS WFC-Home-League-Win' ( standard naming convention)
pattern = r'^Md\d+-[\w\s\.]+-[\w\s\.]+-[\w\s\.]+-[\w\s\.]+-[\w\s\.]+$'
df[df['session_title'].str.match(pattern, na=False)] = df

In [16]:
df.shape

(2720, 99)

In [17]:

# 1. Clean the session title
df['session_title'] = (
    df['session_title']
    .str.strip()
    .str.replace(r'\.+', '', regex=True)         # remove dots like "F.C."
    .str.replace(r'\s*-\s*', '-', regex=True)    # normalize spacing around hyphens
    .str.replace(r'\s+', ' ', regex=True)        # reduce internal whitespace
    .str.title()                                 # ensure proper capitalization
)

# 2. Extract session components: 6 fields + ignore trailing data
regex = re.compile(
    r'^(Md\d+)-'            # Matchday
    r'(.+?)-'               # Club For
    r'(.+?)-'               # Club Against
    r'(.+?)'                # Part 1: location/league/result
    r'[-\s]+(.+?)'          # Part 2: location/league/result
    r'[-\s]+(.+?)'          # Part 3: location/league/result
    r'(?:\s|$)',            # Ignore trailing info
    re.IGNORECASE
)
extracted = df['session_title'].str.extract(regex)

# 3. Assign temporary column names
extracted.columns = [
    'match_day',
    'club_for',
    'club_against',
    'part1',
    'part2',
    'part3'
]

# 4. Define sets to identify
location_set = {'Home', 'Away'}
league_set   = {'League', 'Cup'}
result_set   = {'Win', 'Loss', 'Draw'}

# 5. Assign actual values to correct columns
extracted['location'] = None
extracted['league']   = None
extracted['result']   = None

for i, row in extracted.iterrows():
    for val in [row['part1'], row['part2'], row['part3']]:
        if pd.isna(val): continue
        val_clean = val.strip().title()
        if val_clean in location_set:
            extracted.at[i, 'location'] = val_clean
        elif val_clean in league_set:
            extracted.at[i, 'league'] = val_clean
        elif val_clean in result_set:
            extracted.at[i, 'result'] = val_clean

# 6. Keep only fully matched and valid rows
valid = extracted.dropna(subset=['location', 'league', 'result']).copy()

# 7. Finalize merge
df = df.loc[valid.index].reset_index(drop=True)
valid = valid.drop(columns=['part1', 'part2', 'part3']).reset_index(drop=True)
df = pd.concat([df.reset_index(drop=True), valid], axis=1)


In [18]:
df.shape

(2665, 105)

In [19]:
# Extract player name, club, and position from 'player_name' column

# STEP 1: Fix incorrect entries where second separator is an underscore
# This converts 'First Last - Club_Position' → 'First Last - Club - Position'
def fix_player_name_format(name):
    if re.fullmatch(r'.+ - .+_.+', name):
        return re.sub(r'(.+ - .+?)_(.+)', r'\1 - \2', name)
    return name

df['player_name'] = df['player_name'].apply(fix_player_name_format)

# STEP 2: Extract the fields from the fixed format
player_regex = re.compile(
    r'^(.+?)\s*-\s*'     # Player name
    r'(.+?)\s*-\s*'      # Club
    r'(.+?)$'            # Position
)

player_cols = df['player_name'].str.extract(player_regex)
player_cols.columns = ['p_name', 'player_club_', 'player_position']

# STEP 3: Keep only rows where all 3 groups were found
valid = player_cols.dropna().copy()
df = df.loc[valid.index].reset_index(drop=True)
player_cols = player_cols.loc[valid.index].reset_index(drop=True)

# STEP 4: Add extracted columns back
df = pd.concat([df, player_cols], axis=1)


In [20]:
df.shape

(2647, 108)

In [21]:
# #split the player name and session title columns into columns 
# df[['match_day', 'club_for', 'club_against', 'location', 'league', 'result']] = df['session_title'].str.split('-', n=5, expand=True)
# df[['p_name', 'player_club_', 'player_position']] = df['player_name'].str.split('-', n=2, expand=True)
df = df.drop(['session_title', 'player_name'], axis=1)

# Apply cleaning to each text column again to catch any new text data
cat_cols = [col for col in df.columns if df[col].dtype == 'object']
for col in cat_cols:
    df[col] = df[col].astype(str).apply(clean_text)

In [22]:
#ensure all entries have the proper 'league' value; fix any that may be interchanged with location or result 
mask = df['league'] != 'League'
df.loc[mask, ['location', 'league']] = df.loc[mask, ['league', 'location']].values
df.loc[df['league'] == 'Home', 'league'] = 'League'
df = df.drop('league', axis=1)

In [23]:
#ensure proper location values; switch any interchanged and fill any missed one 
df.loc[df['location'] == 'Draw', ['location', 'result']] = ['Home', 'Draw']
df.loc[df['location'] == 'Wsl', 'location'] = np.nan
df.loc[df['location'] == 'Way', ['location', 'result']] = ['Away', np.nan]


In [24]:
df.shape

(2647, 105)

In [25]:
# list of standard FWSL club names

standard_clubs = [
    'Kampala Queens FC','She Maroons FC','Makerere University WFC',
    'Uganda Martyrs HS WFC','Kawempe Muslim LFC','Wakiso Hill WFC','Olila HS WFC',
    'Rines SS WFC','Amus College WFC','She Corporates FC','Lady Doves FC','FC Tooro Queens','Asubo Ladies FC','St Noa Girls FC'
]

def normalize_name(name):
    # Remove spaces, punctuation, and 'fc', 'sc', etc.
    name = name.lower()
    name = re.sub(r'[^a-z]', '', name)  # keep only letters
    name = name.replace('fc', '').replace('sc', '')
    return name

def best_match(name, club_list, min_score=0.6):
    name_clean = name.strip().lower().replace('.', '')
    norm_name = normalize_name(name_clean)
    # 1. Normalized exact match
    for club in club_list:
        if norm_name == normalize_name(club):
            return club
    # 2. Normalized substring match
    for club in club_list:
        club_norm = normalize_name(club)
        if norm_name in club_norm or club_norm in norm_name:
            return club
    # 3. Token overlap (as before)
    name_tokens = set(name_clean.split())
    best = None
    best_score = 0
    for club in club_list:
        club_tokens = set(club.strip().lower().replace('.', '').split())
        score = len(name_tokens & club_tokens) / max(len(club_tokens), 1)
        if score > best_score:
            best = club
            best_score = score
    return best if best_score >= min_score else name

# Apply to all columns with club names 
for col in ['club_for', 'club_against', 'player_club_']:
    if col in df.columns:
        df[col] = df[col].astype(str).apply(lambda x: best_match(x, standard_clubs))


In [26]:
df.shape

(2647, 105)

In [27]:
for col in ['club_for', 'club_against', 'player_club_']:
    if col in df.columns:
        df[col] = df[col].replace('Uganda Martyrs Hs', 'Uganda Martyrs HS WFC')
        df[col] = df[col].replace("Uganda Martyr'S Hs", 'Uganda Martyrs HS WFC')
        df[col] = df[col].replace("Uganda Martrys", 'Uganda Martyrs HS WFC')
        df[col] = df[col].replace('She Corporate Wfc', 'She Corporates FC')
        df[col] = df[col].replace('Olila High School', 'Olila HS WFC')
        df[col] = df[col].replace('Wakiso Hills', 'Wakiso Hill WFC')
        df[col] = df[col].replace('She Coperates', 'She Corporates FC')

In [28]:
df['player_position'] = df['player_position'].replace('Muslim-Am', 'Am')
df['player_position'] = df['player_position'].replace('F', 'Fwd')

In [29]:
df['player_position'] = df['player_position'].str.lower()

In [30]:
df['general_position'] = df['player_position'].map({
            'gk': 'goalkeepers', 'df': 'defenders', 'mf': 'midfielders', 'am': 'midfielders',
            'fw': 'forwards', 'rb': 'defenders', 'cb': 'defenders', 'lb': 'defenders',
            'rw': 'forwards', 'lw': 'forwards', 'cm': 'midfielders', 'dm': 'midfielders',
            'cd':'defenders','fwd':'forwards','dmc':'midfielders','amc':'midfielders','dc':'defenders',
            'cf':'forwards','mc':'midfielders','lcb':'defenders','md':'midfielders'
        })

In [31]:
df.shape

(2647, 106)

In [32]:
df = df[df['player_position'] != 'gk']

In [33]:
df.shape

(2506, 106)

In [34]:
# Apply cleaning to each text column again to catch any new text data
cat_cols = [col for col in df.columns if df[col].dtype == 'object']
for col in cat_cols:
    df[col] = df[col].astype(str).apply(clean_text)
    df[col] = df[col].astype('category')

In [35]:
zero_frac = (df == 0).mean().sort_values(ascending=False)
sparse = zero_frac[zero_frac > 0.95]
sparse.index.tolist()

# #drop these columns(more than 95% of their values are 0) from the dataset
df = df.drop(columns=sparse.index.tolist())

In [36]:
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns#all numeric columns
num_cols = ['duration','distance_km','sprint_distance_m','power_plays','energy_kcal','impacts',
                   'player_load','top_speed_kmh','distance_per_min_mmin','power_score_wkg','work_ratio','max_acceleration_mss','max_deceleration_mss']# numeric columns of high interest

In [37]:
#Identify the columns with missing data
missing_counts = df.isnull().sum().sort_values(ascending=False)
print(f'Number of columns with missing data: {len(missing_counts[missing_counts > 0])}')
# Check for duplicate rows in the dataset
duplicate_rows = df.duplicated()
print(f"Number of duplicate rows: {duplicate_rows.sum()}")
df = df[~duplicate_rows]  # Remove duplicate rows
print('Duplicates removed')

df = df.drop(columns=['split_start_time', 'split_end_time'])


Number of columns with missing data: 0
Number of duplicate rows: 0
Duplicates removed


In [38]:
df.shape

(2506, 91)

In [39]:
df['split_name'].unique()

['All', '1St Half', '2Nd Half']
Categories (3, object): ['1St Half', '2Nd Half', 'All']

In [40]:
df.head()

,date,split_name,duration,distance_km,sprint_distance_m,power_plays,energy_kcal,impacts,player_load,top_speed_ms,...,deceleration_zone_count:_>_4_mss,match_day,club_for,club_against,location,result,p_name,player_club_,player_position,general_position
0,1970-01-01 00:00:00.000045933,All,95.100000,8.1474,551.961,42,746.0754,6,384.1242,6.9645,...,19,Md1,Kawempe Muslim Lfc,Kampala Queens Fc,Home,Draw,Samalie Nakacwa,Kawempe Muslim Lfc,Rb,Defenders
1,1970-01-01 00:00:00.000045941,All,92.766667,8.1948,371.107,33,738.7179,5,375.8450,7.4401,...,22,Md2,Kawempe Muslim Lfc,Olila Hs Wfc,Away,Win,Samalie Nakacwa,Kawempe Muslim Lfc,Rb,Defenders
2,1970-01-01 00:00:00.000045949,All,95.133333,7.9338,419.094,36,722.1375,8,374.6104,7.6178,...,33,Md3,Kawempe Muslim Lfc,St Noa Girls Fc,Home,Win,Samalie Nakacwa,Kawempe Muslim Lfc,Rb,Defenders
3,1970-01-01 00:00:00.000045955,All,95.966667,7.8497,401.372,36,717.6114,4,374.8868,6.8889,...,13,Md4,Kawempe Muslim Lfc,Uganda Martyrs Hs Wfc,Away,Win,Samalie Nakacwa,Kawempe Muslim Lfc,Rb,Defenders
4,1970-01-01 00:00:00.000045933,All,95.750000,9.2349,607.972,57,840.4633,1,464.7169,8.1334,...,29,Md1,Kawempe Muslim Lfc,Kampala Queens Fc,Home,Draw,Amina Nakato,Kawempe Muslim Lfc,Lb,Defenders


In [41]:
def describe_numeric_columns(df):
    n_desc = df[num_cols].describe().T
    n_desc = n_desc.drop(['25%', '75%'], axis=1)
    n_desc = n_desc.rename(columns={'50%': 'median'})
    display(n_desc.T[num_cols].T)

def describe_categorical_columns(df):
    cat_vars = df.select_dtypes(include='category').columns
    c_desc = df[cat_vars].describe().T
    c_desc['%age'] = c_desc['freq'] / len(df) * 100
    display(c_desc)
    
def plot_numerical_distributions(df):
    plt.figure(figsize=(16, 8))
    for i, col in enumerate(num_cols):
        plt.subplot(4, 4, i + 1)
        sns.histplot(df[col], bins=30, kde=True)
        plt.title(f'Distribution of {col}')
        plt.xlabel(col)
        plt.ylabel('Frequency')
    plt.tight_layout()



In [42]:
df.groupby(['club_for','match_day']).size()

C:\Users\Travail\AppData\Local\Temp\ipykernel_15700\149993504.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(['club_for','match_day']).size()


club_for               match_day
Amus College Wfc       Md1          33
                       Md10          0
                       Md11          0
                       Md2          42
                       Md3           0
                                    ..
Uganda Martyrs Hs Wfc  Md5           0
                       Md6           0
                       Md7          42
                       Md8           0
                       Md9          36
Length: 77, dtype: int64

In [43]:
# Convert top_speed_ms to top_speed_kmh (m/s to km/h: multiply by 3.6)
df['top_speed_kmh'] = df['top_speed_ms'] * 3.6
df = df.drop('top_speed_ms', axis=1)

In [44]:
half2_df = df[df['split_name'] == '2Nd Half']
half2_df = half2_df.drop('split_name', axis=1)

half1_df = df[df['split_name'] == '1St Half']
half1_df = half1_df.drop('split_name', axis=1)

all_df = df[df['split_name'] == 'All']
all_df = all_df.drop('split_name', axis=1)

In [45]:
len(half1_df), len(half2_df), len(all_df)

(835, 835, 836)

In [46]:
merge_keys = ['p_name', 'club_for', 'club_against', 'player_club_','match_day','general_position','player_position','result','location']


half1_df_rn = half1_df.rename(columns=lambda x: x + '_H1' if x not in merge_keys else x)
half2_df_rn = half2_df.rename(columns=lambda x: x + '_H2' if x not in merge_keys else x)
all_df_rn = all_df.rename(columns=lambda x: x + '_ALL' if x not in merge_keys else x)

df_merged = pd.merge(half1_df_rn, half2_df_rn, on=merge_keys, how='outer')
df_merged = pd.merge(df_merged, all_df_rn, on=merge_keys, how='outer')


df_combined = df_merged[merge_keys].copy()

for col in num_cols:
    # Skip metrics that need special handling
    if col not in ['top_speed_kmh', 'distance_per_min_mmin', 'max_acceleration_mss', 
                   'max_deceleration_mss', 'work_ratio', 'power_score_wkg']:
        h1 = f"{col}_H1"
        h2 = f"{col}_H2"
        df_combined[col] = df_merged[h1].fillna(0) + df_merged[h2].fillna(0)

# Top Speed - consider max
df_combined['top_speed_kmh'] = df_merged[['top_speed_kmh_H1', 'top_speed_kmh_H2']].max(axis=1)
# Max Acceleration - consider max
df_combined['max_acceleration_mss'] = df_merged[['max_acceleration_mss_H1', 'max_acceleration_mss_H2']].max(axis=1)
# Max Deceleration - consider max
df_combined['max_deceleration_mss'] = df_merged[['max_deceleration_mss_H1', 'max_deceleration_mss_H2']].max(axis=1)
# distance per minute - recalculate from sum
df_combined['distance_per_min_mmin'] = (df_combined['distance_km'] *1000) / df_combined['duration']
#work ratio - take the one from all if it exists, otherwise put nan - cant be calcualted from halves by defintion
df_combined['work_ratio'] = df_merged['work_ratio_ALL']
#power score - take the one from all if it exists, otherwise calculate from halves-also cant be calculated from halves by definiion
df_combined['power_score_wkg'] = df_merged['power_score_wkg_ALL']

#all other numerical variables - consider sum
# Columns in numeric_cols that are not in num_cols
for col in numeric_cols.difference(num_cols):
    if col not in merge_keys:
        h1 = f"{col}_H1"
        h2 = f"{col}_H2"
        # Only sum if both columns exist in df_merged
        if h1 in df_merged.columns and h2 in df_merged.columns:
            df_combined[col] = df_merged[h1].fillna(0) + df_merged[h2].fillna(0)


In [47]:
match_df = df_combined

In [48]:
# Apply cleaning to each text column
cat_cols = [col for col in match_df.columns if match_df[col].dtype == 'object']
for col in cat_cols:
    match_df[col] = match_df[col].astype(str).apply(clean_text)

In [49]:
match_df.shape

(860, 89)

In [50]:
match_df.groupby(['club_for','match_day']).size()

C:\Users\Travail\AppData\Local\Temp\ipykernel_15700\1871463543.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  match_df.groupby(['club_for','match_day']).size()


club_for               match_day
Amus College Wfc       Md1          11
                       Md10          0
                       Md11          0
                       Md2          14
                       Md3           0
                                    ..
Uganda Martyrs Hs Wfc  Md5           0
                       Md6           0
                       Md7          14
                       Md8           0
                       Md9          12
Length: 77, dtype: int64

In [51]:
# Thresholds
DIST_THRESH = 2   # players should cover at least 2km in a session to count
DURATION_THRESH = 60  # players should be active for at least 60 minutes in a session    

# Extend your active_session flag:
match_df['active_session'] = (
    (match_df['distance_km']    >= DIST_THRESH)  
    & (match_df['duration']  >= DURATION_THRESH)   
)

match_df = match_df[match_df['active_session'] == True]  # Filter main df for active sessions
match_df = match_df.drop('active_session', axis=1)  # Drop the active_session column after filtering

In [52]:
#Calculating Outlier Bounds
q1_dist = match_df['distance_km'].quantile(0.25)
q3_dist = match_df['distance_km'].quantile(0.75)
iqr_dist = q3_dist - q1_dist
lower_bound_dist = q1_dist - 1.5 * iqr_dist
upper_bound_dist = q3_dist + 1.5 * iqr_dist

q1_dur = match_df['duration'].quantile(0.25)
q3_dur = match_df['duration'].quantile(0.75)
iqr_dur = q3_dur - q1_dur
lower_bound_dur = q1_dur - 1.5 * iqr_dur
upper_bound_dur = q3_dur + 1.5 * iqr_dur

#filter out outliers based on the calculated bounds
match_df = match_df[
    (match_df['distance_km'] >= lower_bound_dist) & (match_df['distance_km'] <= upper_bound_dist) &
    (match_df['duration'] >= lower_bound_dur) & (match_df['duration'] <= upper_bound_dur)
]

In [53]:
print(f"Duration bounds: {lower_bound_dur:.2f} to {upper_bound_dur:.2f}")
print(f"Distance bounds: {lower_bound_dist:.2f} to {upper_bound_dist:.2f}")


Duration bounds: 84.57 to 111.77
Distance bounds: 4.48 to 11.81


In [54]:
match_df['duration'].describe()

count    571.000000
mean      97.845563
std        4.887534
min       85.416667
25%       94.866667
50%       97.233333
75%      101.566667
max      110.733333
Name: duration, dtype: float64

In [55]:
# describe_categorical_columns(match_df)

In [56]:
# raw_counts = match_df.groupby(['club_for', 'match_day'])['p_name'].nunique().reset_index(name='raw_unique_players')
# raw_counts[raw_counts['club_for'] == 'Kawempe Muslim Lfc'].sort_values('match_day')


In [57]:
# describe_numeric_columns(match_df)

In [58]:
match_df.groupby(['club_for','match_day']).size()

C:\Users\Travail\AppData\Local\Temp\ipykernel_15700\1871463543.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  match_df.groupby(['club_for','match_day']).size()


club_for               match_day
Amus College Wfc       Md1           8
                       Md10          0
                       Md11          0
                       Md2           8
                       Md3           0
                                    ..
Uganda Martyrs Hs Wfc  Md5           0
                       Md6           0
                       Md7           9
                       Md8           0
                       Md9          10
Length: 77, dtype: int64

In [ ]:
# Save the cleaned and processed DataFrame to a new CSV file
# match_df.to_csv('C:\\FUFA Code\\Match-Analysis\\data\\processed\\FWSL_25_26_mid_season_matches_cleaned.csv',index=False)